# Retinal Screening — Kaggle trainer (thin wrapper)
Clones the repo, installs deps, attaches Kaggle data, runs the config-driven CLI.
Set `CONFIG` below. See `kaggle/README.md` for the required Input datasets.

In [ ]:
REPO_URL = "https://github.com/PraharshNagpure/retinal-ai.git"  # <-- set me
CONFIG   = "configs/dr.yaml"   # configs/rop.yaml | configs/glaucoma.yaml
EXTRA    = ""                   # e.g. "train.epochs=25 model.arch=efficientnet_v2_s"
RUN_DQA  = True

In [ ]:
import os, subprocess
if not os.path.exists('retinal-ai'):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, 'retinal-ai'], check=True)
os.chdir('retinal-ai')
subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('repo + deps ready in', os.getcwd())

In [ ]:
# DR only: extract EyePACS (train.zip -> /tmp/train/train, labels -> /tmp). Skipped otherwise.
import os
if 'dr.yaml' in CONFIG:
    TRAIN_ZIP = '/kaggle/input/diabetic-retinopathy-detection/train.zip.001'
    CSV_ZIP   = '/kaggle/input/diabetic-retinopathy-detection/trainLabels.csv.zip'
    if not os.path.exists('/tmp/train/train'):
        os.system('apt-get install -y p7zip-full >/dev/null 2>&1')
        os.makedirs('/tmp/train', exist_ok=True)
        os.system(f'7z x {TRAIN_ZIP} -o/tmp/train/ -y >/dev/null')
    if not os.path.exists('/tmp/trainLabels.csv'):
        os.system(f'unzip -o -q {CSV_ZIP} -d /tmp/')
    print('EyePACS ready')

In [ ]:
dqa = '--dqa' if RUN_DQA else ''
!python prepare_data.py --config {CONFIG} {dqa}
extra = f'--set {EXTRA}' if EXTRA else ''
!python train.py --config {CONFIG} {extra}

In [ ]:
# Bundle outputs for download (pull weights into the repo via git LFS afterwards).
import shutil
shutil.make_archive('/kaggle/working/results', 'zip', 'results')
print('saved /kaggle/working/results.zip')